# Custom Scrapers — Comprehensive Dry Run

Test **all** remote + data/analyst job sources:
- **15 RSS feeds** (from existing `app/scraper.py` + rssjobs.app, RealWorkFromAnywhere, Jobicy)
- **7 JSON APIs** (Remotive, Jobicy, hiring.cafe, Arbeitnow, Himalayas, WorkingNomads, Jobscollider)

**Setup:** activate `.venv`, then run cells.  
**CLI alternative:** `python dry_run.py`

## 1. Environment check

In [2]:
import sys, json, time
import requests
import feedparser
import pandas as pd
import ipywidgets as w
from IPython.display import display as ipy_display

print(f"Python:     {sys.version}")
print(f"requests:   {requests.__version__}")
print(f"feedparser: {feedparser.__version__}")
print(f"pandas:     {pd.__version__}")
print("All imports OK.")

UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
TIMEOUT = 20

ModuleNotFoundError: No module named 'feedparser'

---
## 2. Define all sources

Every feed URL and API endpoint in one place — pulled from `app/scraper.py` plus new additions.

In [1]:
RSS_FEEDS = {
    # Reliable sources from scraper.py
    "WeWorkRemotely":       "https://weworkremotely.com/remote-jobs.rss",
    "Jobscollider":         "https://jobscollider.com/remote-jobs.rss",
    "Jobscollider (Data)":  "https://jobscollider.com/remote-data-jobs.rss",
    "RemoteOK":             "https://remoteok.com/remote-jobs.rss",
    "Remotive (Data)":      "https://remotive.com/remote-jobs/feed/data",
    "Remotive (AI/ML)":     "https://remotive.com/remote-jobs/feed/ai-ml",
    "Jobspresso":           "https://jobspresso.co/remote-jobs/feed/",
    "Authentic Jobs":       "https://authenticjobs.com/rss/",
    "HN Jobs (hnrss)":     "https://hnrss.org/jobs",
    # New RSS sources
    "rssjobs.app":          "https://rssjobs.app/feeds?keywords=analyst&location=remote",
    "Jobicy RSS":           "https://jobicy.com/jobs-rss-feed",
    "RealWorkFromAnywhere": "https://www.realworkfromanywhere.com/rss.xml",
    # Flaky but worth testing
    "Himalayas RSS":        "https://himalayas.app/jobs/feed",
    "Remote.co":            "https://remote.co/remote-jobs/feed/",
    "Wellfound (analyst)":  "https://wellfound.com/jobs.rss?keywords=data-analyst&remote=true",
}

JSON_APIS = {
    "Remotive API":      {"url": "https://remotive.com/api/remote-jobs",
                           "params": {"category": "data", "search": "analyst", "limit": 50},
                           "jobs_key": "jobs"},
    "Jobicy API":        {"url": "https://jobicy.com/api/v2/remote-jobs",
                           "params": {"count": 50, "tag": "data analyst"},
                           "jobs_key": "jobs"},
    "Arbeitnow API":     {"url": "https://www.arbeitnow.com/api/job-board-api",
                           "params": {},
                           "jobs_key": "data"},
    "Himalayas API":     {"url": "https://himalayas.app/jobs/api",
                           "params": {"q": "analyst", "limit": 50},
                           "jobs_key": "jobs"},
    "hiring.cafe":       {"url": "https://hiring.cafe/api/search-jobs",
                           "params": {"searchQuery": "analyst", "workplaceTypes": "Remote"},
                           "jobs_key": "results",
                           "extra_headers": {"Referer": "https://hiring.cafe/"}},
    "WorkingNomads API": {"url": "https://www.workingnomads.com/api/exposed_jobs/",
                           "params": {},
                           "jobs_key": "_list_"},
    "Jobscollider API":  {"url": "https://jobscollider.com/api/search-jobs",
                           "params": {"title": "data analyst"},
                           "jobs_key": "_auto_"},
}

print(f"RSS feeds:  {len(RSS_FEEDS)}")
print(f"JSON APIs:  {len(JSON_APIS)}")
print(f"Total:      {len(RSS_FEEDS) + len(JSON_APIS)} sources")

RSS feeds:  15
JSON APIs:  7
Total:      22 sources


---
## 3. Fetch all RSS feeds

In [ ]:
rss_results = []

for name, url in RSS_FEEDS.items():
    start = time.time()
    try:
        resp = requests.get(url, timeout=TIMEOUT, headers={"User-Agent": UA})
        elapsed = int((time.time() - start) * 1000)
        if resp.status_code != 200:
            rss_results.append({"source": name, "type": "rss", "status": resp.status_code,
                                "jobs": 0, "ms": elapsed, "error": f"HTTP {resp.status_code}"})
            continue
        feed = feedparser.parse(resp.text)
        count = len(feed.entries)
        sample = ""
        if feed.entries:
            sample = getattr(feed.entries[0], "title", "")[:80]
        rss_results.append({"source": name, "type": "rss", "status": 200,
                            "jobs": count, "ms": elapsed, "sample": sample, "error": ""})
    except Exception as e:
        elapsed = int((time.time() - start) * 1000)
        rss_results.append({"source": name, "type": "rss", "status": 0,
                            "jobs": 0, "ms": elapsed, "error": str(e)[:60]})

rss_df = pd.DataFrame(rss_results)
display(rss_df.style.applymap(lambda v: "color: green" if v > 0 else "color: red",
                              subset=["jobs"]))

---
## 4. Fetch all JSON APIs

In [ ]:
api_results = []

for name, cfg in JSON_APIS.items():
    start = time.time()
    headers = {"User-Agent": UA, "Accept": "application/json"}
    headers.update(cfg.get("extra_headers", {}))
    try:
        resp = requests.get(cfg["url"], params=cfg["params"],
                            headers=headers, timeout=TIMEOUT)
        elapsed = int((time.time() - start) * 1000)
        if resp.status_code != 200:
            api_results.append({"source": name, "type": "json_api", "status": resp.status_code,
                                "jobs": 0, "ms": elapsed, "error": f"HTTP {resp.status_code}"})
            continue
        data = resp.json()
        # Handle different response shapes
        jk = cfg["jobs_key"]
        if jk == "_list_":
            items = data if isinstance(data, list) else []
        elif jk == "_auto_":
            items = data if isinstance(data, list) else data.get("jobs") or data.get("data") or data.get("results") or []
        else:
            items = data.get(jk, []) if isinstance(data, dict) else data
        sample = ""
        if items and isinstance(items[0], dict):
            sample = (items[0].get("title") or items[0].get("jobTitle") or
                      (items[0].get("job_information") or {}).get("title", ""))[:80]
        api_results.append({"source": name, "type": "json_api", "status": 200,
                            "jobs": len(items), "ms": elapsed, "sample": sample, "error": ""})
    except Exception as e:
        elapsed = int((time.time() - start) * 1000)
        api_results.append({"source": name, "type": "json_api", "status": 0,
                            "jobs": 0, "ms": elapsed, "error": str(e)[:60]})

api_df = pd.DataFrame(api_results)
display(api_df.style.applymap(lambda v: "color: green" if v > 0 else "color: red",
                              subset=["jobs"]))

---
## 5. Combined summary

In [ ]:
all_df = pd.concat([rss_df, api_df], ignore_index=True)

working = all_df[all_df["jobs"] > 0].sort_values("jobs", ascending=False)
failed = all_df[all_df["error"].astype(bool)]
empty = all_df[(all_df["jobs"] == 0) & (~all_df["error"].astype(bool))]

print("=" * 55)
print("COMPREHENSIVE DRY RUN SUMMARY")
print("=" * 55)
print(f"Sources tested:  {len(all_df)}")
print(f"Working (PASS):  {len(working)}")
print(f"Empty (0 jobs):  {len(empty)}")
print(f"Errors (FAIL):   {len(failed)}")
print(f"Total jobs:      {all_df['jobs'].sum():,}")
print()
print("Top sources by job count:")
display(working[["source", "type", "jobs", "ms", "sample"]].head(15))

if len(failed) > 0:
    print("\nFailed sources:")
    display(failed[["source", "status", "error"]])

---
## 6. Deep dive — hiring.cafe (richest data)

In [ ]:
hc_kw = w.Text(value="analyst", description="Search", layout=w.Layout(width="50%"))
hc_wp = w.Dropdown(options=["Remote", "Onsite", "Hybrid"], value="Remote", description="Workplace")
hc_limit = w.IntSlider(value=50, min=10, max=200, step=10, description="Max jobs")
hc_out = w.Output()
hc_df = None

def parse_hc(item):
    ji = item.get("job_information") or {}
    vpd = item.get("v5_processed_job_data") or {}
    ec = item.get("enriched_company_data") or {}
    comp = ""
    cur = vpd.get("listed_compensation_currency") or "USD"
    for p in ["yearly", "monthly", "hourly"]:
        lo = vpd.get(f"{p}_min_compensation")
        hi = vpd.get(f"{p}_max_compensation")
        if lo or hi:
            comp = f"{cur} {lo or '?':,}-{hi or '?':,} {p}" if isinstance(lo, (int, float)) else f"{cur} {lo}-{hi} {p}"
            break
    return {
        "title": ji.get("title") or vpd.get("core_job_title", ""),
        "company": ec.get("name") or vpd.get("company_name", ""),
        "location": vpd.get("formatted_workplace_location", ""),
        "workplace": vpd.get("workplace_type", ""),
        "commitment": ", ".join(vpd.get("commitment") or []),
        "seniority": vpd.get("seniority_level", ""),
        "compensation": comp,
        "skills": ", ".join((vpd.get("technical_tools") or [])[:5]),
        "date": vpd.get("estimated_publish_date", ""),
        "url": item.get("apply_url", ""),
        "expired": item.get("is_expired", False),
    }

def fetch_hc(_):
    global hc_df
    with hc_out:
        hc_out.clear_output()
        r = requests.get("https://hiring.cafe/api/search-jobs",
                         params={"searchQuery": hc_kw.value, "workplaceTypes": hc_wp.value},
                         headers={"User-Agent": UA, "Accept": "application/json",
                                  "Referer": "https://hiring.cafe/"}, timeout=30)
        items = r.json().get("results", [])
        jobs = [parse_hc(it) for it in items[:hc_limit.value]]
        active = [j for j in jobs if not j["expired"]]
        print(f"{len(items)} total | {len(active)} active | {len(jobs)-len(active)} expired")
        hc_df = pd.DataFrame(jobs)
        ipy_display(hc_df[["title", "company", "location", "workplace", "seniority", "compensation"]].head(20))

hc_btn = w.Button(description="Fetch hiring.cafe", button_style="primary")
hc_btn.on_click(fetch_hc)
ipy_display(w.VBox([w.HBox([hc_kw, hc_wp]), hc_limit, hc_btn, hc_out]))

---
## 7. Deep dive — rssjobs.app (largest volume)

In [ ]:
rj_kw = w.Text(value="analyst", description="Keywords", layout=w.Layout(width="50%"))
rj_loc = w.Text(value="remote", description="Location", layout=w.Layout(width="50%"))
rj_limit = w.IntSlider(value=50, min=10, max=200, step=10, description="Max jobs")
rj_out = w.Output()
rj_df = None

def fetch_rj(_):
    global rj_df
    url = f"https://rssjobs.app/feeds?keywords={rj_kw.value}&location={rj_loc.value}"
    with rj_out:
        rj_out.clear_output()
        r = requests.get(url, timeout=30, headers={"User-Agent": UA})
        feed = feedparser.parse(r.text)
        print(f"{len(feed.entries)} total entries")
        jobs = []
        for e in feed.entries[:rj_limit.value]:
            t = getattr(e, "title", "")
            company = t.rsplit(" at ", 1)[1].strip() if " at " in t else ""
            title = t.rsplit(" at ", 1)[0].strip() if " at " in t else t
            jobs.append({"title": title, "company": company,
                         "url": getattr(e, "link", ""),
                         "date": getattr(e, "published", "")})
        rj_df = pd.DataFrame(jobs)
        ipy_display(rj_df.head(20))

rj_btn = w.Button(description="Fetch rssjobs", button_style="primary")
rj_btn.on_click(fetch_rj)
ipy_display(w.VBox([w.HBox([rj_kw, rj_loc]), rj_limit, rj_btn, rj_out]))

---
## 8. Export (optional)

In [ ]:
import csv

for label, df in [("all_sources", all_df), ("hiringcafe", hc_df), ("rssjobs", rj_df)]:
    if df is not None and not df.empty:
        path = f"{label}_results.csv"
        df.to_csv(path, quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
        print(f"Saved {len(df)} rows -> {path}")
    else:
        print(f"{label}: nothing to export")